# Sprint 1 — Core API Foundation

Spec: [`sprint-1-tasks.md`](../../litemapper/docs/requirements/sprint-1-tasks.md) — 11 tasks (S1-T00..T10) that shipped the foundational types every other sprint builds on.

| Scope                                                 | Task    |
| ----------------------------------------------------- | ------- |
| `TypePair` readonly struct                            | S1-T01  |
| `MappingStrategy` enum                                | S1-T02  |
| `ITypeTransformer` interface                          | S1-T03  |
| `PropertyLink` + `Blueprint` immutable records        | S1-T06  |
| `MappingScope` class                                  | S1-T06b |
| `ISculptor` + `IMapper<S,D>` interfaces               | S1-T09  |

This notebook exercises those building blocks via the public `SculptorBuilder` → `Forge()` → `ISculptor` lifecycle.


## Setup


In [2]:
#r "../src/SmartMapp.Net/bin/Release/net10.0/SmartMapp.Net.dll"

using SmartMapp.Net;
using SmartMapp.Net.Abstractions;

Console.WriteLine($"Loaded {typeof(SculptorBuilder).Assembly.GetName().Name} v{typeof(SculptorBuilder).Assembly.GetName().Version}");


Loaded SmartMapp.Net v1.0.0.0


## 1. `TypePair` — the identity key for every blueprint

`TypePair` is a readonly struct that uniquely identifies an `(origin, target)` pair. It's the hash key inside every cache in the library — blueprint lookup, inspection cache, projection cache, closed-generic mapper cache.


In [3]:
public sealed class User    { public int Id { get; init; } public string Name { get; init; } = ""; }
public sealed class UserDto { public int Id { get; set; }  public string Name { get; set; } = ""; }

var pair1 = TypePair.Of<User, UserDto>();
var pair2 = new TypePair(typeof(User), typeof(UserDto));

Console.WriteLine($"pair1 == pair2? {pair1 == pair2}");
Console.WriteLine($"origin.target = {pair1.OriginType.Name} → {pair1.TargetType.Name}");
Console.WriteLine($"HashCode stable: {pair1.GetHashCode() == pair2.GetHashCode()}");


pair1 == pair2? True
origin.target = User → UserDto
HashCode stable: True


## 2. `SculptorBuilder` → `Forge()` → `ISculptor` lifecycle

The builder is a one-shot accumulator. You call `.Configure(o => …)` / `.UseBlueprint<T>()` / `.ScanAssembliesContaining<T>()` to queue work, then `.Forge()` freezes the configuration and returns an immutable `ISculptor`. Once forged, the sculptor is thread-safe and can be shared across threads.


In [4]:
var builder  = new SculptorBuilder().Configure(o => o.Bind<User, UserDto>(_ => { }));
var sculptor = builder.Forge();

var user = new User { Id = 1, Name = "Ada" };
var dto  = sculptor.Map<User, UserDto>(user);

Console.WriteLine($"Map result: Id={dto.Id}, Name=\"{dto.Name}\"");
Console.WriteLine($"Sculptor type: {sculptor.GetType().FullName}");
try { _ = builder.Forge(); } catch (InvalidOperationException ex) { Console.WriteLine($"Second Forge() threw {ex.GetType().Name}: {ex.Message}"); }


Map result: Id=1, Name="Ada"
Sculptor type: SmartMapp.Net.Sculptor
Second Forge() threw InvalidOperationException: SculptorBuilder has already been forged — configuration is immutable after Forge() returns.


## 3. `IMapper<TOrigin, TTarget>` — strongly-typed fast path

`ISculptor.Map<S, D>(s)` dispatches by type pair; `IMapper<S, D>` is the closed-generic wrapper the runtime caches per-pair. Calling the mapper directly skips the type-pair lookup dictionary — useful when you only need one pair and want the tightest possible hot loop.


In [5]:
// IMapper<S, D> is the closed-generic fast path — the same IMapper<> that DI would
// register (see Sprint 8). Without DI we reach it through the concrete Sculptor:
var concrete = (Sculptor)sculptor;
IMapper<User, UserDto> mapper = concrete.GetMapper<User, UserDto>();

var src = new User { Id = 42, Name = "Grace" };
var viaSculptor = sculptor.Map<User, UserDto>(src);
var viaMapper   = mapper.Map(src);

Console.WriteLine($"via ISculptor:   Id={viaSculptor.Id}, Name=\"{viaSculptor.Name}\"");
Console.WriteLine($"via IMapper<>:   Id={viaMapper.Id},   Name=\"{viaMapper.Name}\"");
Console.WriteLine($"Mapper instance reused: {object.ReferenceEquals(mapper, concrete.GetMapper<User, UserDto>())}");


via ISculptor:   Id=42, Name="Grace"
via IMapper<>:   Id=42,   Name="Grace"
Mapper instance reused: False


## 4. `Blueprint` + `PropertyLink` — the immutable instruction set

Every `Bind<S, D>` call materialises into a `Blueprint` at forge time. The blueprint holds the target type, strategy, and a `PropertyLink` per target member. Blueprints are immutable — you cannot mutate a forged sculptor, which is what guarantees thread safety.


In [6]:
var config = (ISculptorConfiguration)sculptor;
foreach (Blueprint bp in config.GetAllBlueprints())
{
    Console.WriteLine($"Blueprint: {bp.TypePair.OriginType.Name} → {bp.TypePair.TargetType.Name}");
    Console.WriteLine($"  Strategy      : {bp.Strategy}");
    Console.WriteLine($"  TrackRefs     : {bp.TrackReferences}");
    Console.WriteLine($"  Links         : {bp.Links.Count}");
    foreach (var link in bp.Links)
        Console.WriteLine($"    • {link.TargetMember.Name}  ←  {link.LinkedBy.ConventionName} ({link.LinkedBy.OriginMemberPath})");
}


Blueprint: User → UserDto
  Strategy      : ExpressionCompiled
  TrackRefs     : False
  Links         : 2
    • Id  ←  ExactNameConvention (Id)
    • Name  ←  ExactNameConvention (Name)


## 5. `MappingScope` — per-call state for hooks, providers, and recursion

`MappingScope` threads through every mapping call. It carries:

- A per-call reference dictionary (used by `.TrackReferences()` — see Sprint 4 notebook)
- The current depth (clamped by `.DepthLimit()`)
- Opaque state bags for addons / filters / hooks

You rarely construct one directly; it's surfaced to `IValueProvider<,,>` / `ITypeTransformer<,>` / hook callbacks.


In [7]:
var scope = new MappingScope { MaxDepth = 3 };
Console.WriteLine($"Initial         : CurrentDepth={scope.CurrentDepth}, MaxDepth={scope.MaxDepth}, MaxReached={scope.IsMaxDepthReached}");

var child1 = scope.CreateChild();
Console.WriteLine($"After CreateChild: CurrentDepth={child1.CurrentDepth}");

var child2 = child1.CreateChild();
Console.WriteLine($"Nested           : CurrentDepth={child2.CurrentDepth}");

var child3 = child2.CreateChild();
Console.WriteLine($"Depth-3          : CurrentDepth={child3.CurrentDepth}, MaxReached={child3.IsMaxDepthReached}");

// Scope also carries an arbitrary state bag for filters / addons / hooks.
scope.Items["request-id"] = Guid.NewGuid();
Console.WriteLine($"Items bag        : request-id = {scope.Items["request-id"]}");


Initial         : CurrentDepth=0, MaxDepth=3, MaxReached=False
After CreateChild: CurrentDepth=1
Nested           : CurrentDepth=2
Depth-3          : CurrentDepth=3, MaxReached=True
Items bag        : request-id = 03aed051-8a51-4bd6-ad25-34a0e34ea246


## Next

- **`sprint-02-conventions.ipynb`** — how the convention pipeline auto-links properties without a fluent rule.
